# Day 53 — Custom Loss Function & Model Components
- Build custom loss functions
- Build parameterized custom losses
- Save and load models containing custom components
- Create custom activations
- Create custom initializers
- Create custom regularizers
- Create custom constraints
- Create custom metrics
- Understand the difference between a loss and a metric

## 1. Imports and reproducibility

We use TensorFlow/Keras and NumPy. The examples are intentionally small so they can run quickly on a local Windows/Anaconda setup.


In [1]:
import numpy as np
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

tf.random.set_seed(42)
np.random.seed(42)


TensorFlow version: 2.21.0


## 2. Baseline: built-in MSE

Before creating a custom loss, let's reproduce Mean Squared Error manually.

For each example:

$$
MSE = mean((y_{true} - y_{pred})^2)
$$


In [2]:
y_true = tf.constant([[1.0], [2.0], [3.0]])
y_pred = tf.constant([[1.5], [1.8], [2.5]])

mse_builtin = tf.keras.losses.MeanSquaredError()(y_true, y_pred)
mse_manual = tf.reduce_mean(tf.square(y_true - y_pred))

print("Built-in MSE:", float(mse_builtin))
print("Manual MSE:", float(mse_manual))


Built-in MSE: 0.18000000715255737
Manual MSE: 0.18000000715255737


## 3. Custom loss function

A loss function receives:
- `y_true`: correct target values
- `y_pred`: model predictions

It returns a tensor representing the error.

### Key idea

**Loss → tells the optimizer what it should minimize.**


In [3]:
@tf.keras.utils.register_keras_serializable()
def my_mse(y_true, y_pred):
    error = y_true - y_pred
    return tf.reduce_mean(tf.square(error))

print("Custom MSE:", float(my_mse(y_true, y_pred)))


Custom MSE: 0.18000000715255737


## 4. Custom Huber loss

Huber loss behaves like squared error for small errors and approximately like absolute error for large errors.

For threshold $\delta$:

- Small error: $\frac{1}{2}e^2$
- Large error: $\delta |e| - \frac{1}{2}\delta^2$

This makes it less sensitive to large outliers than MSE.


In [4]:
@tf.keras.utils.register_keras_serializable()
def create_huber(threshold=1.0):
    def huber_fn(y_true, y_pred):
        error = y_true - y_pred
        is_small_error = tf.abs(error) <= threshold

        squared_loss = tf.square(error) / 2.0
        linear_loss = threshold * tf.abs(error) - (threshold ** 2) / 2.0

        return tf.where(is_small_error, squared_loss, linear_loss)

    return huber_fn

huber_1 = create_huber(1.0)
huber_2 = create_huber(2.0)

print("Huber (threshold=1):", huber_1(y_true, y_pred).numpy())
print("Huber (threshold=2):", huber_2(y_true, y_pred).numpy())


Huber (threshold=1): [[0.125     ]
 [0.02000001]
 [0.125     ]]
Huber (threshold=2): [[0.125     ]
 [0.02000001]
 [0.125     ]]


## 5. Train a small regression model with a custom loss

We will create a simple synthetic regression dataset.

The model uses our custom Huber loss.


In [5]:
X = np.random.uniform(-3, 3, size=(500, 1)).astype(np.float32)
y = (3 * X + 2 + np.random.normal(0, 0.8, size=(500, 1))).astype(np.float32)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=[1]),
    tf.keras.layers.Dense(30, activation="relu"),
    tf.keras.layers.Dense(1)
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss=create_huber(1.0),
    metrics=["mae"]
)

history = model.fit(
    X, y,
    epochs=10,
    validation_split=0.2,
    verbose=0
)

print("Final training loss:", history.history["loss"][-1])
print("Final validation loss:", history.history["val_loss"][-1])


Final training loss: 0.30782362818717957
Final validation loss: 0.2779608964920044


## 6. Save and load a model with a custom loss

When a model contains a custom component, Keras needs to know how to reconstruct that component.

Using `register_keras_serializable()` makes custom objects easier to serialize.

We will save the model in the modern `.keras` format.


In [6]:
model_path = "day53_custom_loss_model.keras"
model.save(model_path)

print("Saved:", model_path)


Saved: day53_custom_loss_model.keras


In [7]:
loaded_model = tf.keras.models.load_model(
    model_path,
    custom_objects={"huber_fn": create_huber(1.0)}
)

pred_original = model.predict(X[:3], verbose=0)
pred_loaded = loaded_model.predict(X[:3], verbose=0)

print("Original predictions:")
print(pred_original.ravel())

print("\nLoaded-model predictions:")
print(pred_loaded.ravel())

print("\nPredictions approximately equal:",
      np.allclose(pred_original, pred_loaded, atol=1e-5))


Original predictions:
[-0.26310033 10.075129    6.248276  ]

Loaded-model predictions:
[-0.26310033 10.075129    6.248276  ]

Predictions approximately equal: True


## 7. Custom activation function

An activation transforms a neuron's output.

Here is a simple custom activation similar to a leaky ReLU:

$$
f(z) = max(0.2z, z)
$$

The important idea is:

**Activation → controls how a neuron transforms its output.**


In [8]:
@tf.keras.utils.register_keras_serializable()
def my_activation(z):
    return tf.maximum(0.2 * z, z)

z = tf.constant([-3.0, -1.0, 0.0, 1.0, 3.0])
print("Input:", z.numpy())
print("Custom activation:", my_activation(z).numpy())


Input: [-3. -1.  0.  1.  3.]
Custom activation: [-0.6 -0.2  0.   1.   3. ]


In [9]:
activation_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=[4]),
    tf.keras.layers.Dense(8, activation=my_activation),
    tf.keras.layers.Dense(1)
])

activation_model.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 8)              │            40 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 49 (196.00 B)

 Trainable params: 49 (196.00 B)

 Non-trainable params: 0 (0.00 B)

## 8. Custom initializer

An initializer determines the starting values of trainable weights.

Here we create weights from a normal distribution with a small standard deviation.


In [10]:
@tf.keras.utils.register_keras_serializable()
def my_initializer(shape, dtype=tf.float32):
    return tf.random.normal(
        shape,
        mean=0.0,
        stddev=0.05,
        dtype=dtype
    )

initializer_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=[4]),
    tf.keras.layers.Dense(
        8,
        kernel_initializer=my_initializer,
        activation="relu"
    ),
    tf.keras.layers.Dense(1)
])

weights = initializer_model.layers[0].get_weights()
print("Kernel shape:", weights[0].shape)
print("First few kernel values:", weights[0].ravel()[:10])


Kernel shape: (4, 8)
First few kernel values: [ 0.01637343 -0.04213129  0.01597168 -0.0703776  -0.119403   -0.05196239
 -0.02786616  0.02698535  0.08497161  0.01444683]


## 9. Custom regularizer

A regularizer adds a penalty to the model's total loss.

Example:

$$
Penalty = \lambda \sum w^2
$$

This is conceptually similar to L2 regularization.


In [11]:
@tf.keras.utils.register_keras_serializable()
def my_l2_regularizer(weights):
    return 0.01 * tf.reduce_sum(tf.square(weights))

regularized_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=[4]),
    tf.keras.layers.Dense(
        8,
        activation="relu",
        kernel_regularizer=my_l2_regularizer
    ),
    tf.keras.layers.Dense(1)
])

regularized_model.compile(
    optimizer="adam",
    loss="mse"
)

dummy_X = np.random.randn(16, 4).astype(np.float32)
dummy_y = np.random.randn(16, 1).astype(np.float32)

_ = regularized_model(dummy_X)

print("Number of regularization losses:", len(regularized_model.losses))
print("Regularization loss:", float(regularized_model.losses[0]))


Number of regularization losses: 1
Regularization loss: 0.04505717754364014


## 10. Custom constraint

A constraint restricts trainable weights after updates.

This example clips weights to the interval `[-1, 1]`.

**Constraint → controls the allowed range/structure of weights.**


In [12]:
@tf.keras.utils.register_keras_serializable()
def my_constraint(weights):
    return tf.clip_by_value(weights, -1.0, 1.0)

constraint_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=[4]),
    tf.keras.layers.Dense(
        8,
        activation="relu",
        kernel_constraint=my_constraint
    ),
    tf.keras.layers.Dense(1)
])

constraint_model.compile(
    optimizer="adam",
    loss="mse"
)

constraint_model.fit(dummy_X, dummy_y, epochs=2, verbose=0)

kernel = constraint_model.layers[0].kernel.numpy()

print("Minimum kernel value:", kernel.min())
print("Maximum kernel value:", kernel.max())


Minimum kernel value: -0.6786306
Maximum kernel value: 0.69679254


## 11. Custom metric

A metric measures performance during training.

Unlike the loss, a metric is primarily for monitoring.

Here we implement Mean Absolute Error manually:

$$
MAE = mean(|y_{true} - y_{pred}|)
$$


In [13]:
@tf.keras.utils.register_keras_serializable()
def my_mae(y_true, y_pred):
    return tf.reduce_mean(tf.abs(y_true - y_pred))

print("Custom MAE:", float(my_mae(y_true, y_pred)))
print("Built-in MAE:",
      float(tf.keras.metrics.MeanAbsoluteError()(y_true, y_pred)))


Custom MAE: 0.4000000059604645
Built-in MAE: 0.4000000059604645


In [14]:
metric_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=[1]),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1)
])

metric_model.compile(
    optimizer="adam",
    loss="mse",
    metrics=[my_mae]
)

metric_history = metric_model.fit(
    X, y,
    epochs=5,
    validation_split=0.2,
    verbose=0
)

print("Available history keys:")
print(metric_history.history.keys())


Available history keys:
dict_keys(['loss', 'my_mae', 'val_loss', 'val_my_mae'])


## 12. Putting several custom components together

This example combines:
- custom activation
- custom initializer
- custom regularizer
- custom constraint
- custom loss
- custom metric

This is the main idea of today's chapter section: **Keras is extensible.**


In [15]:
combined_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=[1]),
    tf.keras.layers.Dense(
        16,
        activation=my_activation,
        kernel_initializer=my_initializer,
        kernel_regularizer=my_l2_regularizer,
        kernel_constraint=my_constraint
    ),
    tf.keras.layers.Dense(1)
])

combined_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss=create_huber(1.0),
    metrics=[my_mae]
)

combined_history = combined_model.fit(
    X, y,
    epochs=5,
    validation_split=0.2,
    verbose=0
)

print("Final loss:", combined_history.history["loss"][-1])
print("Final MAE:", combined_history.history["my_mae"][-1])


Final loss: 0.42395687103271484
Final MAE: 0.7695599794387817


# Quick Revision Sheet



| Component | Main purpose |
|---|---|
| **Loss** | Guides optimization |
| **Metric** | Measures performance |
| **Activation** | Transforms neuron output |
| **Initializer** | Chooses starting weights |
| **Regularizer** | Adds a penalty to the loss |
| **Constraint** | Restricts trainable weights |

### Easy memory trick

- **Loss = Learn**
- **Metric = Measure**
- **Initializer = Initial state**
- **Regularizer = Penalty**
- **Constraint = Restriction**
- **Activation = Transformation**



## Custom components

```text
Custom Loss
    ↓
Defines what the optimizer minimizes

Custom Activation
    ↓
Defines how neuron outputs are transformed

Custom Initializer
    ↓
Defines how weights start

Custom Regularizer
    ↓
Adds a penalty for undesirable weights

Custom Constraint
    ↓
Restricts weights after updates

Custom Metric
    ↓
Measures model performance
```
